In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [3]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [4]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [6]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [40]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    
    prompt = f"""
    
Please solve the following task

{test_case["task"]}
"""
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output


In [ ]:
def grade_by_model(test_case, output):
    # Create evaluation prompt
    eval_prompt = f"""
You are an expert code reviewer. Evaluate this AI-generated solution.

Task: {test_case["task"]}
Solution: {output}

Provide your evaluation as a single valid JSON object with these keys:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement  
- "reasoning": A concise explanation of your assessment
- "score": A number between 1-10

Important JSON formatting rules:
- Respond with ONLY the JSON object — no markdown fences, no extra text before or after
- All backslashes must be escaped as \\\\ (e.g. a regex \\d must be written as \\\\d)
- All double quotes inside string values must be escaped as \\"
- Do not include literal newlines inside string values — use \\n instead
- Do not quote large blocks of code verbatim in "reasoning" — paraphrase or reference it briefly instead
"""
    
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [68]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    #Grading
    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

In [74]:
from statistics import mean

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [75]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)
results = run_eval(dataset)

Average score: 6.5


In [76]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 URI Parser\n\n```python\nfrom urllib.parse import urlparse\nfrom typing import Tuple, Optional\n\ndef parse_s3_uri(s3_uri: str) -> Tuple[str, str, str]:\n    \"\"\"\n    Parse an AWS S3 object key and extract bucket name, prefix, and file name.\n    \n    Args:\n        s3_uri: Full S3 URI in format 's3://bucket-name/path/to/file.txt'\n        \n    Returns:\n        Tuple of (bucket_name, prefix, file_name)\n        - bucket_name: The S3 bucket name\n        - prefix: The directory path (empty string if file is in root)\n        - file_name: The file name only\n        \n    Raises:\n        ValueError: If the URI is not a valid S3 URI\n        \n    Example:\n        >>> parse_s3_uri('s3://my-bucket/path/to/file.txt')\n        ('my-bucket', 'path/to', 'file.txt')\n        \n        >>> parse_s3_uri('s3://my-bucket/file.txt')\n        ('my-bucket', '', 'file.txt')\n    \"\"\"\n    # Validate and parse the URI\n    if not s3_uri.startswith('s3://'):\n     